# Advanced 02 — Cryptographic Delegation, Capabilities & Verifiable Provenance

Scenario: Alice delegates limited claims authority to an autonomous claims agent, which delegates read-only research to a specialist and obtains a one-use payment capability after approval.


In [ ]:
from datetime import datetime,timedelta,timezone
import base64,json,uuid,hashlib,copy
import pandas as pd
import networkx as nx
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
NOW=datetime.now(timezone.utc)
def canonical(x):return json.dumps(x,sort_keys=True,separators=(",",":")).encode()
def b64(x):return base64.urlsafe_b64encode(x).decode().rstrip("=")
def ub64(x):return base64.urlsafe_b64decode(x+"="*(-len(x)%4))


## 1 — Generate an issuer key

In [ ]:
root_private=Ed25519PrivateKey.generate()
root_public=root_private.public_key()


## 2 — Create a root capability

In [ ]:
root_claims={
 "jti":str(uuid.uuid4()),"issuer":"user:alice","subject":"agent:claims",
 "audience":"claims-api","actions":["claim.read","claim.update","knowledge.search"],
 "resources":["claim:483","kb:claims"],"task":"task:77",
 "issued_at":NOW.isoformat(),"expires_at":(NOW+timedelta(hours=1)).isoformat(),"depth":0
}


## 3 — Sign

In [ ]:
root_token={"claims":root_claims,"signature":b64(root_private.sign(canonical(root_claims)))}
root_token


## 4 — Verify

In [ ]:
root_public.verify(ub64(root_token["signature"]),canonical(root_token["claims"]))
print("signature valid")


## 5 — Tampering detection

In [ ]:
tampered=copy.deepcopy(root_token);tampered["claims"]["actions"].append("payment.create")
try:
    root_public.verify(ub64(tampered["signature"]),canonical(tampered["claims"]))
    print("BAD")
except Exception:
    print("tampering detected")


## 6 — Validate audience, action, resource and expiry

In [ ]:
def validate(c,audience,action,resource,now):
    if c["audience"]!=audience:return False,"AUDIENCE"
    if action not in c["actions"]:return False,"ACTION"
    if resource not in c["resources"]:return False,"RESOURCE"
    if now>=datetime.fromisoformat(c["expires_at"]):return False,"EXPIRED"
    return True,"VALID"
validate(root_claims,"claims-api","claim.read","claim:483",NOW)


## 7 — Attenuation

In [ ]:
def attenuate(parent,subject,actions,resources,expires):
    if not set(actions).issubset(parent["actions"]):raise ValueError("action escalation")
    if not set(resources).issubset(parent["resources"]):raise ValueError("resource escalation")
    if expires>datetime.fromisoformat(parent["expires_at"]):raise ValueError("lifetime expansion")
    return {"jti":str(uuid.uuid4()),"issuer":parent["subject"],"subject":subject,
      "audience":parent["audience"],"actions":list(actions),"resources":list(resources),
      "task":parent["task"],"issued_at":NOW.isoformat(),"expires_at":expires.isoformat(),
      "parent":parent["jti"],"depth":parent["depth"]+1}
child=attenuate(root_claims,"agent:research",["claim.read","knowledge.search"],
                ["claim:483","kb:claims"],NOW+timedelta(minutes=15))
child


## 8 — Escalation fails

In [ ]:
try:
    attenuate(root_claims,"agent:research",["claim.delete"],["claim:483"],NOW+timedelta(minutes=10))
except ValueError as e: print(e)


## 9 — Sign child grant

In [ ]:
agent_private=Ed25519PrivateKey.generate();agent_public=agent_private.public_key()
child_token={"claims":child,"signature":b64(agent_private.sign(canonical(child)))}
agent_public.verify(ub64(child_token["signature"]),canonical(child))


## 10 — Delegation provenance graph

In [ ]:
g=nx.DiGraph()
g.add_edge("user:alice","agent:claims",capability=root_claims["jti"])
g.add_edge("agent:claims","agent:research",capability=child["jti"])
list(g.edges(data=True))


## 11 — Chain validation invariant

In [ ]:
assert set(child["actions"]).issubset(root_claims["actions"])
assert set(child["resources"]).issubset(root_claims["resources"])
assert datetime.fromisoformat(child["expires_at"])<=datetime.fromisoformat(root_claims["expires_at"])


## 12 — Revocation

In [ ]:
revoked={root_claims["jti"]}
def not_revoked(c): return c["jti"] not in revoked and c.get("parent") not in revoked
not_revoked(child)


## 13 — Replay/JTI cache

In [ ]:
seen=set()
def consume_once(c):
    if c["jti"] in seen:return False
    seen.add(c["jti"]);return True
consume_once(child),consume_once(child)


## 14 — Macaroon-style caveats

In [ ]:
caveats=[
 ("audience","claims-api"),
 ("operation","claim.read"),
 ("resource","claim:483"),
 ("expires_before",(NOW+timedelta(minutes=10)).isoformat())
]
pd.DataFrame(caveats,columns=["caveat","value"])


## 15 — Real Macaroon exercise

With `pymacaroons`, create a root Macaroon and add first-party caveats for:
- target service;
- operation;
- claim ID;
- expiry;
- task ID.

Then verify every caveat against server-side context.

Add an approval-service third-party caveat as an extension exercise.


## 16 — Biscuit-style attenuation model

In [ ]:
biscuit_blocks=[
 {"block":"authority","facts":["right(claim:483,read)","right(claim:483,update)"]},
 {"block":"attenuation-1","checks":["operation == read"]},
 {"block":"attenuation-2","checks":["resource == claim:483"]},
 {"block":"attenuation-3","checks":["time < expiry"]}
]
biscuit_blocks


## 17 — Biscuit exercise

Use a Biscuit implementation in your language/runtime to reproduce the preceding blocks.

Observe that:
- the root authority block establishes rights;
- downstream blocks add checks;
- downstream holders cannot rewrite the authority block;
- sealing prevents further attenuation.

Compare this with the custom signed-envelope implementation.


## 18 — OAuth Token Exchange request

In [ ]:
token_exchange={
 "grant_type":"urn:ietf:params:oauth:grant-type:token-exchange",
 "subject_token":"USER_TOKEN",
 "subject_token_type":"urn:ietf:params:oauth:token-type:access_token",
 "actor_token":"AGENT_TOKEN",
 "actor_token_type":"urn:ietf:params:oauth:token-type:access_token",
 "resource":"https://claims-api",
 "scope":"claim.read"
}
token_exchange


## 19 — Actor chain

In [ ]:
exchanged_claims={"sub":"user:alice","aud":"https://claims-api",
 "scope":"claim.read","act":{"sub":"agent:research","act":{"sub":"agent:claims"}}}
exchanged_claims


## 20 — Delegation vs impersonation

In [ ]:
print("Delegation preserves subject=user:alice and actor=agent:claims")
print("Impersonation would make the actor appear as the subject; use only when intentionally required.")


## 21 — Generate DPoP key

In [ ]:
dpop_private=Ed25519PrivateKey.generate();dpop_public=dpop_private.public_key()


## 22 — Simplified DPoP proof

In [ ]:
proof_claims={"htm":"POST","htu":"https://claims-api/payments",
 "iat":int(NOW.timestamp()),"jti":str(uuid.uuid4())}
proof={"claims":proof_claims,"signature":b64(dpop_private.sign(canonical(proof_claims)))}
dpop_public.verify(ub64(proof["signature"]),canonical(proof["claims"]))


## 23 — Bind proof to access token hash

In [ ]:
access_token="opaque-access-token"
ath=b64(hashlib.sha256(access_token.encode()).digest())
proof_claims["ath"]=ath
ath


## 24 — DPoP production note

The previous cells teach the cryptographic idea, not a full RFC 9449 implementation.

A production implementation must follow RFC 9449 JOSE requirements and validate:
- proof type/header and algorithm;
- public JWK;
- signature;
- `htm`;
- `htu`;
- `iat`;
- unique `jti`;
- `ath`;
- nonce when required;
- access-token confirmation/key binding.

Prefer a standards-compliant OAuth library.


## 25 — MCP resource binding

In [ ]:
mcp={"audience":"https://mcp.claims.example"}
downstream={"audience":"https://claims-api.internal"}
assert mcp["audience"]!=downstream["audience"]
print("MCP token must not be passed through as downstream API token.")


## 26 — Provenance event

In [ ]:
events=[]
def append_event(event):
    prev=events[-1]["hash"] if events else ""
    h=hashlib.sha256(canonical({"event":event,"previous_hash":prev})).hexdigest()
    events.append({"event":event,"previous_hash":prev,"hash":h})
append_event({"type":"issue","jti":root_claims["jti"],"issuer":"user:alice"})
append_event({"type":"delegate","jti":child["jti"],"parent":root_claims["jti"]})
events


## 27 — Detect audit-chain tampering

In [ ]:
def verify_events(rows):
    prev=""
    for r in rows:
        expected=hashlib.sha256(canonical({"event":r["event"],"previous_hash":prev})).hexdigest()
        if expected!=r["hash"]:return False
        prev=r["hash"]
    return True
verify_events(events)


## 28 — Policy integration

In [ ]:
crypto_facts={"signature_valid":True,"chain_valid":True,"sender_bound":True}
current_policy={"workload_approved":True,"risk":"low","relationship_valid":True}
allow=all(crypto_facts.values()) and all([
 current_policy["workload_approved"],
 current_policy["risk"]=="low",
 current_policy["relationship_valid"]
])
allow


## 29 — Adversarial corpus

In [ ]:
attacks=["tampered_payload","wrong_signature","expired","wrong_audience",
"action_escalation","resource_escalation","lifetime_expansion","revoked_parent",
"replay","wrong_dpop_key","token_substitution","mcp_token_passthrough",
"broken_provenance","capability_laundering","confused_deputy"]
pd.DataFrame({"attack":attacks})


## 30 — Payment capability after HITL

In [ ]:
payment_cap={
 "jti":str(uuid.uuid4()),"issuer":"approval-service","subject":"agent:claims",
 "audience":"payments-api","actions":["payment.create"],"resources":["claim:483"],
 "constraints":{"max_amount":750,"currency":"CAD","max_uses":1},
 "task":"task:77","expires_at":(NOW+timedelta(minutes=5)).isoformat()
}
payment_cap


## 31 — Enforce payment capability

In [ ]:
def payment_allowed(cap,amount,currency,claim,now):
    return (cap["audience"]=="payments-api"
      and "payment.create" in cap["actions"]
      and claim in cap["resources"]
      and amount<=cap["constraints"]["max_amount"]
      and currency==cap["constraints"]["currency"]
      and now<datetime.fromisoformat(cap["expires_at"]))
payment_allowed(payment_cap,700,"CAD","claim:483",NOW)


# Capstone

Build a complete secure delegation path:

```text
Alice
  → Claims Agent
  → Research Agent
  → MCP Knowledge Server
```

and a separate HITL path:

```text
Claims Agent
  → Approval Service
  → one-use Payment Capability
  → Payments API
```

Requirements:

1. every credential is audience-bound;
2. child authority is a strict subset of parent authority;
3. expiries cannot expand;
4. delegation depth is bounded;
5. root and child signatures are verified;
6. parent revocation invalidates descendants;
7. high-risk credentials are short-lived;
8. payment capability is one-use;
9. replay is detected;
10. bearer credentials never enter prompts/traces;
11. MCP credentials are not passed through downstream;
12. sender constraint is used for sensitive APIs;
13. current policy is evaluated after cryptographic verification;
14. provenance records every issuance/delegation;
15. adversarial tests prove the invariants.


# Review questions

1. What is the difference between identity and capability?
2. Why is attenuation important for multi-agent systems?
3. What makes a child grant an escalation?
4. How do Macaroon caveats work?
5. What is Biscuit offline attenuation?
6. Why does Biscuit block scoping matter?
7. Why is Biscuit not an authentication protocol?
8. How does RFC 8693 represent delegation?
9. What is the difference between `act` and `may_act`?
10. Why should an agent not impersonate a human by default?
11. How does DPoP reduce token replay?
12. How do mTLS-bound tokens differ from DPoP?
13. Why must tokens be audience/resource-bound?
14. Why does MCP prohibit token passthrough?
15. How should offline credentials be revoked?
16. What is capability laundering?
17. How does cryptographic provenance complement authorization policy?
18. Why should cryptographic verification occur before policy evaluation?
